# Resumen — Sesgo, Correlación y Análisis Integrado

> **Para el TP4 Integrador de Estadística Descriptiva**  
> Skewness, coeficiente de correlación de Pearson y Spearman, heatmaps y scatter plots.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sst

url = 'https://drive.google.com/uc?export=download&id=1leo0WjSbpwWOSTMUmCSMPsch3VindqYe'
df = pd.read_csv(url)

bins   = [0, 18.5, 25, 30, float('inf')]
labels = ['Bajo peso', 'Normal', 'Sobrepeso', 'Obesidad']
df['bmi_category'] = pd.cut(df['bmi'], bins=bins, labels=labels, right=False)
df['smoker_num']   = (df['smoker'] == 'yes').astype(int)

print(f'Dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head(3)


---
## 1. Sesgo (Skewness)

El sesgo mide **la asimetría** de una distribución:

| Valor de skewness | Forma de la distribución | Cola larga |
|---|---|---|
| ≈ 0 | Simétrica (normal) | — |
| > 0 (positivo) | Cola a la **derecha** | Valores extremos altos |
| < 0 (negativo) | Cola a la **izquierda** | Valores extremos bajos |
| \|skew\| > 1 | Sesgo considerable | — |

**Consecuencia práctica:** cuando el sesgo es alto, la media está más alejada de la mediana y se ve más afectada por los valores extremos. En ese caso, **preferir la mediana**.


In [ ]:
cols = ['charges', 'bmi', 'age']
print(f'{'Variable':10s}  {'Skewness':>10s}  {'Interpretación'}')
for col in cols:
    sk = df[col].skew()
    interp = 'simétrica' if abs(sk) < 0.5 else ('sesgo + (cola dcha)' if sk > 0 else 'sesgo - (cola izq)')
    print(f'{col:10s}  {sk:>10.4f}  {interp}')

print()
# Visualización
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cols):
    ax.hist(df[col], bins=30, color='steelblue', edgecolor='white')
    ax.axvline(df[col].mean(),   color='red',    linestyle='--', linewidth=1.5,
               label=f'Media: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange', linestyle='-',  linewidth=1.5,
               label=f'Mediana: {df[col].median():.1f}')
    ax.set_title(f'{col}  (skew={df[col].skew():.2f})')
    ax.legend(fontsize=8)
plt.suptitle('Sesgo: cuando skew > 0, la cola va a la DERECHA y media > mediana')
plt.tight_layout()
plt.show()


---
## 2. Correlación

La correlación mide **la fuerza y dirección de la relación lineal** entre dos variables.

### Coeficiente de Pearson (r)
- Rango: **−1 a +1**
- Solo mide relaciones **lineales**
- Requiere variables **cuantitativas continuas**
- Sensible a outliers

### Coeficiente de Spearman (ρ)
- También rango: **−1 a +1**  
- Mide relaciones **monótonas** (no necesariamente lineales)
- Válido para variables **ordinales** o cuando la relación es no lineal
- Robusto ante outliers

| Valor de r o ρ | Fuerza |
|---|---|
| 0.00 – 0.10 | Despreciable |
| 0.10 – 0.30 | Débil |
| 0.30 – 0.50 | Moderada |
| 0.50 – 0.70 | Fuerte |
| 0.70 – 1.00 | Muy fuerte |


In [ ]:
# Correlación de Pearson
numericas = ['age', 'bmi', 'children', 'charges', 'smoker_num']
corr = df[numericas].corr()
print('Correlaciones con charges (Pearson):')
print(corr['charges'].drop('charges').sort_values(ascending=False).round(3))


In [ ]:
# Heatmap — la visualización estándar para matrices de correlación
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax
)
ax.set_title('Matriz de correlación de Pearson')
plt.tight_layout()
plt.show()

print('Cómo leer el heatmap:')
print('  Rojo oscuro  → correlación positiva fuerte (r cercano a +1)')
print('  Azul oscuro  → correlación negativa fuerte (r cercano a -1)')
print('  Blanco/crema → sin correlación lineal (r ≈ 0)')


### 2.1 Spearman para variables ordinales

Cuando una variable es **ordinal** (como `bmi_category`: Bajo peso < Normal < Sobrepeso < Obesidad), hay que codificarla numéricamente y usar **Spearman** en lugar de Pearson.


In [ ]:
orden = {'Bajo peso': 0, 'Normal': 1, 'Sobrepeso': 2, 'Obesidad': 3}
df['bmi_category_num'] = df['bmi_category'].map(orden)

r_pearson,  p1 = sst.pearsonr( df['bmi_category_num'].dropna(), df.loc[df['bmi_category_num'].notna(), 'charges'])
r_spearman, p2 = sst.spearmanr(df['bmi_category_num'].dropna(), df.loc[df['bmi_category_num'].notna(), 'charges'])

print(f'Pearson  r = {r_pearson:.4f}  (p={p1:.4f})')
print(f'Spearman r = {r_spearman:.4f}  (p={p2:.4f})')
print()
print('Usamos Spearman porque bmi_category es ORDINAL.')
print('Pearson requiere variables cuantitativas continuas.')


### 2.2 Scatter plot — visualizar la correlación

El scatter plot permite ver:
1. **Dirección** (positiva / negativa)
2. **Forma** (lineal / curvilínea)
3. **Fuerza** (puntos apretados = correlación fuerte)
4. **Grupos** (usando `hue` para colorear por categoría)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df,
                ax=ax, alpha=0.5, palette='Set1')
ax.set_title('BMI vs Gastos médicos — separado por condición de fumador')
ax.set_xlabel('BMI')
ax.set_ylabel('Gastos (USD)')
plt.tight_layout()
plt.show()

print('Observación: hay DOS nubes separadas.')
print('Dentro de los fumadores: BMI y charges tienen correlación positiva visible.')
print('En no fumadores: la relación es mucho más débil.')


---
## 3. Multicolinealidad

Cuando dos **features predictoras** tienen alta correlación entre sí (|r| > 0.7), decimos que hay **multicolinealidad**.

**Problema:** en modelos predictivos, incluir dos variables muy correlacionadas es redundante y puede inestabilizar el modelo.  
**Solución:** revisar el heatmap y quedarse con solo una de las dos variables altamente correlacionadas.

> En el dataset de seguros, las features están relativamente poco correlacionadas entre sí, por lo que no hay multicolinealidad problemática.


---
## 4. Resumen visual de todo el flujo de análisis

```
1. Cargar datos → describe() → entender forma y rango
2. Histogramas → calcular skewness → ¿es simétrica?
3. Outliers → IQR o z-score según la forma
4. Correlación → heatmap de Pearson (o Spearman si hay ordinales)
5. Scatter con hue → ¿la relación cambia por grupo?
6. Análisis segmentado → groupby + estadísticas por subgrupo
```

| Pregunta analítica | Herramienta |
|---|---|
| ¿Cómo se distribuye X? | Histograma + media/mediana/skew |
| ¿Hay valores atípicos? | Boxplot + IQR o z-score |
| ¿X e Y están relacionadas? | Correlación Pearson + scatter |
| ¿La relación es igual en todos los grupos? | Scatter con hue / correlación por segmento |
| ¿Qué variable predice mejor Y? | Ordenar correlaciones con Y |
